In [7]:
import pandas as pd


df = pd.read_csv('../data/HUPA0001P.csv', sep=';', parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)
df['carb_input'] = df['carb_input'].fillna(0.0)
df['time'] = pd.to_datetime(df['time'])

# time
#
# glucose
#
# calories
#
# heart_rate
#
# steps
#
# basal_rate
#
# bolus_volume_delivered
#
# carb_input

# [ Past: Time T-1 ]            [ Current: Time T ]
#
#  GL_State_Prev  ------------>  GL_State
#                                   |
#                                   >
#  TUN_State_Prev ------------>  TUN_State <---- TC_Hour
#                                   |
#                                   >
#                                SN_State


# (Time current:TC, time_since_last: TSL, Glucose level: GL) -----> (Time until next:TUN)
#                  ---->   (Size of next: SN)

import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('../data/HUPA0001P.csv', sep=';', parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)

df['carb_input'] = df['carb_input'].fillna(0.0)
df['TC_Hour'] = df['time'].dt.hour
df['glucose'] = df['glucose'].ffill().bfill()
df["GL_State"] = pd.cut(df['glucose'], bins=[-1, 70, 140, 999], labels=[1, 2, 3]).astype(int)

df['last_meal_time'] = df.apply(lambda row: row['time'] if row['carb_input'] > 0 else pd.NaT, axis=1).ffill()
df['hours_since_last'] = (df['time'] - df['last_meal_time']).dt.total_seconds() / 3600
df['hours_since_last'] = df['hours_since_last'].fillna(999)
df['TP_State'] = pd.cut(df['hours_since_last'], bins=[-1, 2, 4, 999], labels=[1, 2, 3]).astype(int)

df['next_meal_time'] = df.apply(lambda row: row['time'] if row['carb_input'] > 0 else pd.NaT, axis=1).bfill()
df['next_meal_amount'] = df.apply(lambda row: row['carb_input'] if row['carb_input'] > 0 else pd.NA, axis=1).bfill()
df['hours_until_next'] = (df['next_meal_time'] - df['time']).dt.total_seconds() / 3600

df['GL_State_Prev'] = df['GL_State'].shift(1)
df['TUN_State_Prev'] = pd.cut(df['hours_until_next'].shift(1), bins=[-0.1, 1, 3, 999], labels=[1, 2, 3])

train_df = df.dropna(subset=['hours_until_next', 'GL_State_Prev', 'TUN_State_Prev']).copy()
train_df['TUN_State'] = pd.cut(train_df['hours_until_next'], bins=[-0.1, 1, 3, 999], labels=[1, 2, 3]).astype(int)
train_df['SN_State'] = pd.cut(train_df['next_meal_amount'], bins=[0, 3, 999], labels=[1, 2]).astype(int)
train_df['GL_State_Prev'] = train_df['GL_State_Prev'].astype(int)
train_df['TUN_State_Prev'] = train_df['TUN_State_Prev'].astype(int)

train_df = train_df.reset_index(drop=True)

features = ['GL_State_Prev', 'TUN_State_Prev', 'TC_Hour', 'GL_State', 'TP_State']
targets = ['TUN_State', 'SN_State']
all_cols = features + targets

#split
tscv = TimeSeriesSplit(n_splits=5)

index = 1
tun_accuracies = []
sn_accuracies = []

for train_index, test_index in tscv.split(train_df):
    train_fragments = train_df.iloc[train_index]
    test_fragments = train_df.iloc[test_index]

    model = DiscreteBayesianNetwork([
        ('GL_State_Prev', 'GL_State'),
        ('TUN_State_Prev', 'TUN_State'),
        ('TC_Hour', 'TUN_State'),
        ('GL_State', 'TUN_State'),
        ('TP_State', 'TUN_State'),
        ('TUN_State', 'SN_State'),
    ])

    model.fit(
        train_fragments[all_cols],
        estimator=BayesianEstimator,
        equivalent_sample_size=10
    )

    X_test = test_fragments[features]
    y_test_tun = test_fragments['TUN_State'].values
    y_test_sn = test_fragments['SN_State'].values

    predictions = model.predict(X_test)

    acc_tun = accuracy_score(y_test_tun, predictions['TUN_State'])
    acc_sn = accuracy_score(y_test_sn, predictions['SN_State'])

    tun_accuracies.append(acc_tun)
    sn_accuracies.append(acc_sn)

    print(f"Train indices: [{train_index[0]} to {train_index[-1]}] (Size: {len(train_fragments)})")
    print(f"Test indices:  [{test_index[0]} to {test_index[-1]}] (Size: {len(test_fragments)})\n")

    print("--- TUN_State (Time Until Next Meal) ---")
    print(f"Accuracy: {acc_tun:.4f}")
    print("Classification report:")
    print(classification_report(y_test_tun, predictions['TUN_State']))
    print("\n")

    print("--- SN_State (Size of Next Meal) ---")
    print(f"Accuracy: {acc_sn:.4f}")
    print("Classification Report:")
    print(classification_report(y_test_sn, predictions['SN_State']))
    print("-" * 60 + "\n")

    index += 1



INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'GL_State_Prev': 'N', 'TUN_State_Prev': 'N', 'TC_Hour': 'N', 'GL_State': 'N', 'TP_State': 'N', 'TUN_State': 'N', 'SN_State': 'N'}


  0%|          | 0/131 [00:00<?, ?it/s]

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'GL_State_Prev': 'N', 'TUN_State_Prev': 'N', 'TC_Hour': 'N', 'GL_State': 'N', 'TP_State': 'N', 'TUN_State': 'N', 'SN_State': 'N'}


Train indices: [0 to 677] (Size: 678)
Test indices:  [678 to 1355] (Size: 678)

--- TUN_State (Time Until Next Meal) ---
Accuracy: 0.8864
Classification report:
              precision    recall  f1-score   support

           1       0.94      0.62      0.75       104
           2       0.97      0.81      0.88       192
           3       0.85      0.99      0.92       382

    accuracy                           0.89       678
   macro avg       0.92      0.81      0.85       678
weighted avg       0.90      0.89      0.88       678



--- SN_State (Size of Next Meal) ---
Accuracy: 0.8614
Classification Report:
              precision    recall  f1-score   support

           1       0.86      1.00      0.93       584
           2       0.00      0.00      0.00        94

    accuracy                           0.86       678
   macro avg       0.43      0.50      0.46       678
weighted avg       0.74      0.86      0.80       678

----------------------------------------------------

  0%|          | 0/148 [00:00<?, ?it/s]

...

  0%|          | 0/123 [00:00<?, ?it/s]

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'GL_State_Prev': 'N', 'TUN_State_Prev': 'N', 'TC_Hour': 'N', 'GL_State': 'N', 'TP_State': 'N', 'TUN_State': 'N', 'SN_State': 'N'}


Train indices: [0 to 2711] (Size: 2712)
Test indices:  [2712 to 3389] (Size: 678)

--- TUN_State (Time Until Next Meal) ---
Accuracy: 0.9233
Classification report:
              precision    recall  f1-score   support

           1       1.00      0.56      0.72        52
           2       0.98      0.73      0.84       109
           3       0.91      1.00      0.95       517

    accuracy                           0.92       678
   macro avg       0.96      0.76      0.84       678
weighted avg       0.93      0.92      0.92       678



--- SN_State (Size of Next Meal) ---
Accuracy: 0.6947
Classification Report:
              precision    recall  f1-score   support

           1       0.69      1.00      0.82       471
           2       0.00      0.00      0.00       207

    accuracy                           0.69       678
   macro avg       0.35      0.50      0.41       678
weighted avg       0.48      0.69      0.57       678

-------------------------------------------------

  0%|          | 0/98 [00:00<?, ?it/s]

Train indices: [0 to 3389] (Size: 3390)
Test indices:  [3390 to 4067] (Size: 678)

--- TUN_State (Time Until Next Meal) ---
Accuracy: 0.9572
Classification report:
              precision    recall  f1-score   support

           1       0.89      0.89      0.89        91
           2       0.97      0.88      0.93       155
           3       0.97      1.00      0.98       432

    accuracy                           0.96       678
   macro avg       0.94      0.92      0.93       678
weighted avg       0.96      0.96      0.96       678



--- SN_State (Size of Next Meal) ---
Accuracy: 0.6976
Classification Report:
              precision    recall  f1-score   support

           1       0.70      1.00      0.82       473
           2       0.00      0.00      0.00       205

    accuracy                           0.70       678
   macro avg       0.35      0.50      0.41       678
weighted avg       0.49      0.70      0.57       678

-------------------------------------------------